In [ ]:
import os
import json
import random
from tqdm import tqdm

# Paths
CONVO_PATH = "data/conversations" 
META_PATH  = "data/metadata"

OUTPUT_PATH = "challenges_solutions_data"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Helpers
def load_json(path):
    """Safely load JSON files, skipping malformed ones."""
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        print(f"Invalid JSON skipped: {path}")
        return None
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None


def extract_client_text(turns):
    """
    Extract text from client-only turns. If turns are raw strings or missing speakers,
    fall back to concatenation of all turn text.
    """
    client_only = []

    for t in turns:
        # Case 1: structured turn {"speaker": "...", "text": "..."}
        if isinstance(t, dict):
            if t.get("speaker") == "client":
                client_only.append(t.get("text", "").strip())

        # Case 2: raw string turn -> no speaker distinction
        elif isinstance(t, str):
            continue  # skip here; fallback will handle this case

    # If we successfully extracted structured client turns
    if client_only:
        return " ".join(client_only).strip()

    # Otherwise fallback: concatenate all text regardless of structure
    fallback_text = []

    for t in turns:
        if isinstance(t, dict):
            fallback_text.append(t.get("text", "").strip())
        elif isinstance(t, str):
            fallback_text.append(t.strip())

    return " ".join(fallback_text).strip()


# Build Challenge + Solution Data
all_challenges = []
all_solutions  = []

conversation_files = sorted([f for f in os.listdir(CONVO_PATH) if f.endswith("_conversation.json")])

for convo_file in tqdm(conversation_files, desc="Processing dataset"):

    # Load conversation 
    convo_path = os.path.join(CONVO_PATH, convo_file)
    convo_data = load_json(convo_path)
    if not convo_data:
        continue

    turns = convo_data.get("full_conversation", [])
    input_text = extract_client_text(turns)

    # Link metadata 
    meta_file = convo_file.replace("_conversation.json", "_metadata.json")
    meta_path = os.path.join(META_PATH, meta_file)
    meta = load_json(meta_path)
    if not meta:
        continue

    # ---------- Extract true cues 
    true_cues = meta.get("client_profile", {}).get("exhibited_behaviors", [])
    if not isinstance(true_cues, list):
        true_cues = []

    # Add Challenge Entry
    all_challenges.append({
        "id": convo_file,
        "task_type": "cue_detection",
        "turns": turns,
        "input_text": input_text,
        "source": "ground_truth"
    })

    # Add Solution Entry
    all_solutions.append({
        "id": convo_file,
        "correct_output": {
            "true_cues": sorted(true_cues),
            "severity": None,        # future SEAL or rule classifier
            "domains": ["trauma"]    # placeholder - will improve later
        },
        "source": "ground_truth"
    })

# Save to JSON
with open(os.path.join(OUTPUT_PATH, "challenges.json"), "w", encoding="utf-8") as f:
    json.dump(all_challenges, f, indent=2, ensure_ascii=False)

with open(os.path.join(OUTPUT_PATH, "solutions.json"), "w", encoding="utf-8") as f:
    json.dump(all_solutions, f, indent=2, ensure_ascii=False)

len(all_challenges), len(all_solutions)


Processing dataset:  33%|███▎      | 976/3000 [00:00<00:00, 9731.31it/s]

Invalid JSON skipped: data/conversations/1_P5_conversation.json


Processing dataset: 100%|██████████| 3000/3000 [00:00<00:00, 7869.04it/s]


(2999, 2999)